# استعادة المشروع وإعادة تشغيل dbt — أمل خوتاني

ضمن **برنامج هندسة البيانات الحديثة لأنظمة الذكاء الاصطناعي — Modern Data Engineering for AI Systems (SDA-DSC-214)** لدى [أكاديمية سدايا](https://github.com/SDAIAAcademy). #SDAIAAcademy

مواد الدورة ودوالها: **ميعاد المري — Meaad Al-Marri**. البيانات اصطناعية.

هذا ملحق أدلة للاب 03، أُنجز بعد الأيام الخمسة لاستعادة أدلة dbt المنفصلة. نُقلت خليتا الاستعادة وإعادة التشغيل من `amal_khotani (4).ipynb`، بترقيم المصدر 88 و89 الذي يبدأ من صفر، مع شيفرتهما ومخرجاتهما وأرقام تنفيذهما الأصلية. لم يُعد التنفيذ أثناء إعداد هذا الملف. بصمة الدفتر المصدر: `d98252b1b47df0fdedcc8b4c5e264c0aed10014426148185ff3c4d1c6963300a`.

**سبب الاستعادة:** عرض البحث السابق `DBT_FILES_NOT_FOUND`، وتوقفت المحاولة التالية قبل تشغيل dbt لأن المشروع لم يكن موجودًا في جلسة Colab. هاتان المحاولتان محفوظتان في خليتي المصدر 86 و87؛ يعرض هذا الملحق الخليتين اللتين عالجتا المشكلة ونجحتا. لا تعني معالجة الخطأ أن المحاولة الأولى كانت ناجحة.

**قبل إعادة التشغيل:** يتطلب هذا المسار Colab مع Python 3.11 ونسخة `day05_handoff.zip`. تنشئ خلية الاستعادة نسخة مستودع جديدة وتثبت Java 17 ومتطلبات Spark، ثم يستدعي مشغّل dbt الأصلي في عملية Python/Spark جديدة. لا يعدّل المشغّل جدول Silver المستمر؛ يعتمد على نسخ معزولة من لقطات Bronze. فحص freshness يستخدم توقيتات الاستقبال الأصلية، وقد يرفض مدخلات قديمة عند إعادة التشغيل لاحقًا؛ لا تُعدّل التوقيتات لتجاوز الفحص.

أرقام تنفيذ الخلايا المعروضة ليست متسلسلة: عداد خلية الاستعادة فارغ في الملف المرفوع رغم وجود مخرجاتها؛ بقي كما ورد. خلية dbt تحمل العداد 4. الدليل المكمل هو تقرير dbt وأوامره الأصلية في الأرشيف المحفوظ.


## استعادة اليوم الخامس وتهيئة البيئة


In [ ]:
from pathlib import Path
from google.colab import files
import io, os, sys, json, zipfile, subprocess, tempfile

print("Python:", sys.version.split()[0])

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        "الدورة تحتاج Python 3.11. أرسلي صورة هذه الرسالة "
        "لنضبط إصدار الجلسة قبل المتابعة."
    )

print("اختاري ملف day05_handoff.zip")
uploaded = files.upload()

zip_names = [
    name for name in uploaded
    if name.lower().endswith(".zip")
]
if len(zip_names) != 1:
    raise ValueError("ارفعي ملف day05_handoff.zip وحده.")

# Restore into a new folder.
ROOT = (
    Path(tempfile.mkdtemp(prefix="masar_restore_", dir="/content"))
    / "course"
)

print("Preparing the course repository...", flush=True)
subprocess.run(
    [
        "git", "clone", "--depth", "1", "--branch", "develop",
        "https://github.com/amalkhotani/masar-modern-data-engineering.git",
        str(ROOT),
    ],
    env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    check=True,
    timeout=180,
)

with zipfile.ZipFile(io.BytesIO(uploaded[zip_names[0]])) as bundle:
    for name in bundle.namelist():
        if (
            not name.startswith("outputs/")
            or ".." in Path(name).parts
            or "\\" in name
        ):
            raise ValueError("Unexpected archive path: " + name)

    if "outputs/day01_bronze_success.json" not in bundle.namelist():
        raise ValueError("ملف ZIP لا يحتوي على مؤشر مساحة العمل.")

    if bundle.testzip() is not None:
        raise ValueError("ZIP integrity check failed")

    bundle.extractall(ROOT)

# Prepare Java 17.
java_home = next(
    (
        p for p in Path("/usr/lib/jvm").glob("*17*")
        if (p / "bin/java").is_file()
    ),
    None,
)

if java_home is None:
    print("Installing Java 17...", flush=True)
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "openjdk-17-jre-headless"],
        check=True,
    )
    java_home = next(
        p for p in Path("/usr/lib/jvm").glob("*17*")
        if (p / "bin/java").is_file()
    )

os.environ["JAVA_HOME"] = str(java_home)
os.environ["PATH"] = str(java_home / "bin") + os.pathsep + os.environ["PATH"]

print("Installing course requirements...", flush=True)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet",
        "-r", str(ROOT / "requirements-runtime.txt"),
    ],
    check=True,
)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from masar.workspace import completed_bronze_workspace, require_fixed_dataset
from masar.runtime import require_environment
from masar.native_contracts import validate_stage_result

SOURCE = ROOT / "data/masar-small-v1"
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)

# Verify the saved Day 5 reports.
for stage, filename in [
    ("lab07_gold_recovery", "day05_recovery.json"),
    ("lab08_serving", "day05_serving_latest.json"),
]:
    report = json.loads(
        (WORK / "reports" / filename).read_text(encoding="utf-8")
    )
    validate_stage_result(stage, report)
    print("Restored report verified:", stage)

print("Workspace:", WORK)
print("DAY05_RESTORED")


Python: 3.11.13
اختاري ملف day05_handoff.zip


Saving day05_handoff.zip to day05_handoff.zip
Preparing the course repository...
Installing Java 17...
Installing course requirements...
Restored report verified: lab07_gold_recovery
Restored report verified: lab08_serving
Workspace: /content/masar_restore_ozff5t3x/course/outputs/day01_bronze_xgnqdyt3
DAY05_RESTORED


## إعادة تشغيل dbt والتحقق وحفظ الأرشيف


In [4]:
from pathlib import Path
from google.colab import files
import os, sys, json, subprocess, tempfile, zipfile

if globals().get("ROOT") is None:
    raise RuntimeError("شغّلي خلية الاستعادة أولًا حتى يظهر DAY05_RESTORED.")

dbt_root = Path(ROOT).resolve()

print("Installing dbt requirements...", flush=True)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "-r", str(dbt_root / "requirements-dbt.txt"),
    ],
    check=True,
)

# Run dbt with a fresh Spark process.
dbt_env = os.environ.copy()
for key in ("PYSPARK_GATEWAY_PORT", "PYSPARK_GATEWAY_SECRET"):
    dbt_env.pop(key, None)

runner = """
from pathlib import Path
import json, sys

root = Path(sys.argv[1])
sys.path.insert(0, str(root / "src"))

from masar.dbt_lab import run_dbt_lab

report, report_path = run_dbt_lab(root)

print(json.dumps({
    "status": report["status"],
    "report": str(report_path),
    "error": report.get("error"),
}, indent=2), flush=True)

if report["status"] != "PASSED_DBT_NATIVE":
    raise RuntimeError(report.get("error") or "dbt did not pass")

Path(sys.argv[2]).write_text(str(report_path), encoding="utf-8")
"""

print("Running dbt... قد يستغرق عدة دقائق.", flush=True)

with tempfile.TemporaryDirectory(prefix="masar_dbt_recovery_") as temp:
    result_path = Path(temp) / "report_path.txt"

    subprocess.run(
        [
            sys.executable, "-u", "-c", runner,
            str(dbt_root), str(result_path),
        ],
        cwd=dbt_root,
        env=dbt_env,
        check=True,
    )

    dbt_path = Path(result_path.read_text(encoding="utf-8"))

dbt_report = json.loads(dbt_path.read_text(encoding="utf-8"))
dbt_workspace = dbt_path.parent.parent

for phase in dbt_report["phases"]:
    print(
        phase["phase"],
        "rows:", phase["rows"],
        "fare SAR:", phase["total_fare_sar"],
    )

# Verify the generated documentation.
docs = dbt_workspace / "dbt/commands/documentation/target"

for name in ("index.html", "manifest.json", "catalog.json"):
    path = docs / name
    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(f"Missing documentation: {name}")

# Save the report, documentation and outputs.
archive = (
    dbt_root / "outputs"
    / f"dbt_evidence_{dbt_report['run_id'][:8]}.zip"
)

with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(dbt_workspace.rglob("*")):
        if path.is_file():
            bundle.write(path, path.relative_to(dbt_root).as_posix())

with zipfile.ZipFile(archive) as bundle:
    if bundle.testzip() is not None:
        raise RuntimeError("ZIP integrity check failed")

print("DBT_EVIDENCE_SAVED")
print("File:", archive.name)
files.download(str(archive))

Installing dbt requirements...
Running dbt... قد يستغرق عدة دقائق.
base rows: 72 fare SAR: 1794.60
rerun rows: 72 fare SAR: 1794.60
late rows: 75 fare SAR: 1875.60
late_replay rows: 75 fare SAR: 1875.60
DBT_EVIDENCE_SAVED
File: dbt_evidence_83b24dcb.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## الأدلة الناتجة

- الحالة في التقرير الأصلي: `PASSED_DBT_NATIVE`.
- المساحة: `outputs/dbt_validation_tedk3wzl`.
- المراحل: base ثم rerun ثم late ثم late_replay؛ الأعداد 72، 72، 75، 75؛ والأجور 1794.60، 1794.60، 1875.60، 1875.60 ريالًا.
- التوثيق: `dbt/commands/documentation/target/index.html` مع `manifest.json` و`catalog.json`؛ 6 نماذج و3 مصادر.
- احتفظ بالأرشيف `dbt_evidence_83b24dcb.zip` إلى جانب `day05_handoff.zip`. الدفتر لا يحتوي ملفات Delta أو ملفات توثيق dbt داخله.

التفاصيل والبصمات في [فهرس الأدلة](../EVIDENCE_INDEX.md). لا تمثل هذه الإعادة تشغيلًا جديدًا للأيام الخمسة أو تقييمًا للمقرر.
